# M12 HAR-RV-J — QuantBook Research Notebook

Replication du modele HAR-RV-J (Andersen, Bollerslev, Diebold 2007 "Roughing It Up")
dans l'environnement QuantConnect Cloud.

**Objectif** : verifier que le verdict BEATS (p=7.9e-7, 64/84 wins) se reproduit
avec les donnees QC Cloud crypto.

**Phases** :
1. Data sourcing via QuantBook (BTC, ETH, SOL, LTC, XRP, ADA, DOT)
2. Realized Variance reconstruction from hourly returns
3. HAR-RV-J fit (walk-forward 5-fold expanding OLS)
4. DM test vs HAR Classic baseline
5. Kelly sizing backtest
6. Verdict matrix coin x horizon

**Reference locale** : `scripts/m12_har_rv_j.py` (BEATS p=7.9e-7, 84 combos)

In [ ]:
# Section 1 — QuantBook Initialization & Data Sourcing
# Execution: QC Cloud uniquement (QuantBook kernel)

from AlgorithmImports import *
from QuantConnect.Research import QuantBook
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

qb = QuantBook()
print(f"QuantBook initialized. User: {qb.UserId}")

## 1.1 — Crypto Universe Setup

M12 utilise 7 paires crypto avec donnees horaires.
QC Cloud fournit `Crypto` via Coinbase/Bitstamp avec resolution horaire.

In [ ]:
# Crypto symbols — QC Cloud format
COINS = {
    "BTC": ("BTCUSD", Market.COINBASE, CryptoType.SPOT),
    "ETH": ("ETHUSD", Market.COINPLACE, CryptoType.SPOT),
    # SOL, LTC, XRP, ADA, DOT may have limited history on QC
    # Fallback to BTC-only if others unavailable
}

crypto_symbols = {}
for ticker, (symbol, market, ctype) in COINS.items():
    try:
        sym = qb.AddCrypto(symbol, Resolution.Hour, market)
        crypto_symbols[ticker] = sym.Symbol
        print(f"{ticker}: {sym.Symbol} added")
    except Exception as e:
        print(f"{ticker}: UNAVAILABLE — {e}")

print(f"\nTotal crypto symbols: {len(crypto_symbols)}")

## 1.2 — Hourly Data Download

HAR-RV-J necessite des returns intra-day pour calculer la Realized Variance.
Resolution.Hour = 24 returns/jour → RV journaliere.

In [ ]:
# Download hourly history (2018-01-01 to present for BTC)
START = datetime(2018, 1, 1)
END = datetime(2025, 12, 31)

hourly_data = {}
for ticker, sym in crypto_symbols.items():
    try:
        df = qb.History(sym, START, END, Resolution.Hour)
        if len(df) > 0:
            hourly_data[ticker] = df
            print(f"{ticker}: {len(df)} hourly bars ({df.index[0][1].date()} to {df.index[-1][1].date()})")
        else:
            print(f"{ticker}: 0 bars returned")
    except Exception as e:
        print(f"{ticker}: History error — {e}")

print(f"\nCoins with data: {list(hourly_data.keys())}")

## Section 2 — Realized Variance Reconstruction

From hourly log-returns, compute daily Realized Variance and Bipower Variation.
Jump component: J_t = max(RV_t - mu * BPV_t, 0) where mu = pi/2 ≈ 0.6 (Huang-Tauchen).

In [ ]:
# Section 2 — RV, BPV, Jump computation
MU_HUANG_TAUCHEN = np.pi / 2  # ≈ 1.5708, but convention is 0.6 for scaled version
# Note: original script uses mu=0.6 (Huang-Tauchen threshold)
MU = 0.6

def compute_daily_rv(hourly_closes: pd.Series) -> pd.Series:
    """Daily Realized Variance from hourly log-returns."""
    log_ret = np.log(hourly_closes / hourly_closes.shift(1)).dropna()
    daily_rv = log_ret.groupby(log_ret.index.date).apply(lambda x: np.sum(x**2))
    daily_rv.index = pd.to_datetime(daily_rv.index)
    daily_rv.index.name = "date"
    daily_rv.name = "RV"
    return daily_rv

def compute_daily_bpv(hourly_closes: pd.Series) -> pd.Series:
    """Daily Bipower Variation (Barndorff-Nielsen & Shephard 2004)."""
    log_ret = np.log(hourly_closes / hourly_closes.shift(1)).dropna()
    abs_ret = np.abs(log_ret)
    # BPV = (pi/2) * sum(|r_{i-1}| * |r_i|) for consecutive intraday returns
    def daily_bpv(group):
        r = abs_ret.loc[group.index]
        if len(r) < 2:
            return np.nan
        return (np.pi / 2) * np.sum(r[:-1].values * r[1:].values)
    daily_bpv = log_ret.groupby(log_ret.index.date).apply(daily_bpv)
    daily_bpv.index = pd.to_datetime(daily_bpv.index)
    daily_bpv.index.name = "date"
    daily_bpv.name = "BPV"
    return daily_bpv

def compute_jumps(rv: pd.Series, bpv: pd.Series) -> pd.Series:
    """Jump component J_t = max(RV_t - mu * BPV_t, 0)."""
    aligned = pd.concat([rv.rename("rv"), bpv.rename("bpv")], axis=1).dropna()
    jumps = np.maximum(aligned["rv"] - MU * aligned["bpv"], 0.0)
    jumps.index.name = "date"
    jumps.name = "J"
    return jumps

print("RV/BPV/Jump functions defined.")
print(f"Huang-Tauchen mu = {MU}")

In [ ]:
# Compute RV, BPV, Jumps for each coin
daily_data = {}

for ticker, df in hourly_data.items():
    # Extract close prices — QC History returns MultiIndex (Symbol, Time)
    if isinstance(df.index, pd.MultiIndex):
        closes = df["close"].droplevel(0)
    else:
        closes = df["close"]
    
    rv = compute_daily_rv(closes)
    bpv = compute_daily_bpv(closes)
    jumps = compute_jumps(rv, bpv)
    
    daily_data[ticker] = {
        "rv": rv,
        "bpv": bpv,
        "jumps": jumps,
        "n_days": len(rv),
    }
    print(f"{ticker}: {len(rv)} days, RV mean={rv.mean():.6f}, Jumps>0: {(jumps > 0).sum()}/{len(jumps)}")

## Section 3 — HAR-RV-J Model

HAR-RV-J regression with 6 features (RV d/w/m + Jump d/w/m):
```
log(RV_{t+1}) = b0 + b_d*log(RV_t) + b_w*log(RV_w) + b_m*log(RV_m)
              + b_dj*J_t + b_wj*J_w + b_mj*J_m + e
```
Walk-forward 5-fold expanding OLS, refit every 22 days.

In [ ]:
# Section 3 — HAR-RV-J model implementation

def har_rv_j_features(rv: pd.Series, jumps: pd.Series) -> pd.DataFrame:
    """Build HAR-RV-J features: RV d/w/m (log) + Jump d/w/m (raw)."""
    log_rv = np.log(rv.clip(lower=1e-12))
    rv_d = log_rv.shift(1)
    rv_w = log_rv.shift(1).rolling(5, min_periods=5).mean()
    rv_m = log_rv.shift(1).rolling(22, min_periods=22).mean()
    j_d = jumps.shift(1)
    j_w = jumps.shift(1).rolling(5, min_periods=5).mean()
    j_m = jumps.shift(1).rolling(22, min_periods=22).mean()
    return pd.DataFrame({
        "rv_d": rv_d, "rv_w": rv_w, "rv_m": rv_m,
        "j_d": j_d, "j_w": j_w, "j_m": j_m,
    })

def har_classic_features(rv: pd.Series) -> pd.DataFrame:
    """Build HAR Classic features: RV d/w/m (log) only."""
    log_rv = np.log(rv.clip(lower=1e-12))
    rv_d = log_rv.shift(1)
    rv_w = log_rv.shift(1).rolling(5, min_periods=5).mean()
    rv_m = log_rv.shift(1).rolling(22, min_periods=22).mean()
    return pd.DataFrame({"rv_d": rv_d, "rv_w": rv_w, "rv_m": rv_m})

def ols_fit(X, y):
    """OLS with intercept."""
    X_aug = np.column_stack([np.ones(len(X)), X])
    coef, *_ = np.linalg.lstsq(X_aug, y, rcond=None)
    return coef

def ols_predict(coef, X):
    X_aug = np.column_stack([np.ones(len(X)), X])
    return X_aug @ coef

print("HAR-RV-J and HAR Classic feature builders defined.")

In [ ]:
# Walk-forward evaluation
N_SPLITS = 5
REFIT_EVERY = 22
HORIZONS = [1, 5, 10]

def walk_forward_comparison(
    rv: pd.Series, jumps: pd.Series, horizon: int = 1,
    n_splits: int = N_SPLITS, refit_every: int = REFIT_EVERY
) -> dict:
    """Walk-forward expanding OLS for HAR-RV-J vs HAR Classic.
    Returns forecasts, actuals, and per-day Kelly positions for both models.
    """
    log_rv = np.log(rv.clip(lower=1e-12))
    target = log_rv.shift(-horizon).dropna()
    
    feats_j = har_rv_j_features(rv, jumps).dropna()
    feats_c = har_classic_features(rv).dropna()
    
    common = target.index.intersection(feats_j.index).intersection(feats_c.index)
    target = target.loc[common]
    feats_j = feats_j.loc[common]
    feats_c = feats_c.loc[common]
    
    n = len(target)
    fold_size = n // (n_splits + 1)
    
    forecasts_j, forecasts_c, actuals = [], [], []
    coef_j_history, coef_c_history = [], []
    
    for k in range(1, n_splits + 1):
        train_end = k * fold_size
        test_start = train_end
        test_end = min(train_end + fold_size, n)
        
        if test_start >= n:
            break
        
        # Expanding window
        X_j_train = feats_j.iloc[:train_end].values
        y_train = target.iloc[:train_end].values
        X_c_train = feats_c.iloc[:train_end].values
        
        coef_j = ols_fit(X_j_train, y_train)
        coef_c = ols_fit(X_c_train, y_train)
        
        # Predict with refit_every spacing
        for i in range(test_start, test_end):
            if (i - test_start) % refit_every == 0 and i > test_start:
                X_j_train = feats_j.iloc[:i].values
                y_train = target.iloc[:i].values
                X_c_train = feats_c.iloc[:i].values
                coef_j = ols_fit(X_j_train, y_train)
                coef_c = ols_fit(X_c_train, y_train)
            
            forecasts_j.append(ols_predict(coef_j, feats_j.iloc[i:i+1].values)[0])
            forecasts_c.append(ols_predict(coef_c, feats_c.iloc[i:i+1].values)[0])
            actuals.append(target.iloc[i])
    
    return {
        "forecast_j": np.array(forecasts_j),
        "forecast_c": np.array(forecasts_c),
        "actual": np.array(actuals),
        "mse_j": np.mean((np.array(forecasts_j) - np.array(actuals))**2),
        "mse_c": np.mean((np.array(forecasts_c) - np.array(actuals))**2),
    }

print(f"Walk-forward function defined. {N_SPLITS} folds, refit every {REFIT_EVERY} days.")

## Section 4 — Run Walk-Forward for Each Coin × Horizon

Replicate the M12 experimental protocol: 7 coins × 3 horizons = 21 combos.
Compare HAR-RV-J vs HAR Classic via sign-test on paired forecast errors.

In [ ]:
# Section 4 — Full sweep
results = []

for ticker, data in daily_data.items():
    for h in HORIZONS:
        try:
            wf = walk_forward_comparison(data["rv"], data["jumps"], horizon=h)
            
            # Sign test: does HAR-RV-J beat HAR Classic?
            err_j = (wf["forecast_j"] - wf["actual"])**2
            err_c = (wf["forecast_c"] - wf["actual"])**2
            j_wins = np.sum(err_j < err_c)
            n_total = len(err_j)
            p_value = stats.binomtest(j_wins, n_total, alternative='greater').pvalue
            
            result = {
                "coin": ticker,
                "horizon": h,
                "n_obs": n_total,
                "j_wins": j_wins,
                "win_rate": j_wins / n_total,
                "p_value": p_value,
                "mse_j": wf["mse_j"],
                "mse_c": wf["mse_c"],
                "mse_ratio": wf["mse_j"] / wf["mse_c"] if wf["mse_c"] > 0 else np.nan,
            }
            results.append(result)
            verdict = "BEATS" if p_value < 0.05 else "NO BEATS"
            print(f"{ticker} h={h:2d}: {j_wins}/{n_total} ({result['win_rate']:.1%}) p={p_value:.4f} {verdict}")
        except Exception as e:
            print(f"{ticker} h={h:2d}: ERROR — {e}")

df_results = pd.DataFrame(results)
print(f"\n=== Summary ===")
if len(df_results) > 0:
    beats = df_results[df_results["p_value"] < 0.05]
    print(f"Total combos: {len(df_results)}")
    print(f"BEATS: {len(beats)}/{len(df_results)}")
    print(f"Mean win rate: {df_results['win_rate'].mean():.1%}")
    print(f"Mean MSE ratio (J/Classic): {df_results['mse_ratio'].mean():.4f}")
else:
    print("No results — check data availability above.")

## Section 5 — Kelly Sizing Backtest

Using HAR-RV-J vol forecasts to size positions via Kelly criterion.
Threshold band: go long when forecast vol < recent vol (buy the dip in vol).
Kelly cap = 1.0, fee = 50bps.

In [ ]:
# Section 5 — Kelly backtest
KELLY_CAP = 1.0
FEE_BPS = 50

def kelly_backtest(
    rv: pd.Series, jumps: pd.Series, horizon: int = 1,
    kelly_cap: float = KELLY_CAP, fee_bps: float = FEE_BPS
) -> dict:
    """Simple Kelly threshold-band backtest using HAR-RV-J forecasts."""
    log_rv = np.log(rv.clip(lower=1e-12))
    feats = har_rv_j_features(rv, jumps).dropna()
    target = log_rv.shift(-1).dropna()
    
    common = target.index.intersection(feats.index)
    n = len(common)
    fold_size = n // 6  # 5 folds + 1 initial train
    
    positions = []
    returns = []
    
    for k in range(1, 6):
        train_end = k * fold_size
        test_end = min(train_end + fold_size, n)
        
        X_train = feats.loc[common[:train_end]].values
        y_train = target.loc[common[:train_end]].values
        coef = ols_fit(X_train, y_train)
        
        for i in range(train_end, test_end):
            forecast_vol = np.exp(ols_predict(coef, feats.loc[common[i:i+1]].values)[0])
            recent_vol = rv.loc[common[i-22]:common[i]].mean() if i >= 22 else rv.loc[common[:i]].mean()
            
            # Threshold band: long when forecast < recent (low vol = good)
            if forecast_vol < recent_vol:
                kelly_frac = min(KELLY_CAP, forecast_vol / recent_vol)
            else:
                kelly_frac = 0.0
            
            # Realized return
            actual_log_ret = log_rv.loc[common[i]] - log_rv.loc[common[i-1]] if i > 0 else 0
            pnl = kelly_frac * actual_log_ret - (fee_bps / 10000) * abs(kelly_frac)
            
            positions.append(kelly_frac)
            returns.append(pnl)
    
    returns = np.array(returns)
    cum_ret = np.cumprod(1 + returns)
    
    return {
        "cumulative_return": cum_ret[-1] if len(cum_ret) > 0 else 1.0,
        "sharpe": np.mean(returns) / np.std(returns) * np.sqrt(365) if np.std(returns) > 0 else 0,
        "max_dd": 1 - np.min(cum_ret / np.maximum.accumulate(cum_ret)) if len(cum_ret) > 0 else 0,
        "n_trades": len(returns),
        "positions": positions,
        "returns": returns,
    }

# Run Kelly backtest for each coin
kelly_results = []
for ticker, data in daily_data.items():
    try:
        kb = kelly_backtest(data["rv"], data["jumps"])
        kelly_results.append({"coin": ticker, **kb})
        print(f"{ticker}: Sharpe={kb['sharpe']:.3f}, CumRet={kb['cumulative_return']:.2f}, MaxDD={kb['max_dd']:.1%}")
    except Exception as e:
        print(f"{ticker}: ERROR — {e}")

print(f"\nKelly backtest complete for {len(kelly_results)} coins.")

## Section 6 — Verdict Matrix & Comparison

Aggregate results into a verdict matrix: coin × horizon.
Compare with local Python verdict (BEATS 64/84, p=7.9e-7).

In [ ]:
# Section 6 — Final verdict
print("=" * 60)
print("M12 HAR-RV-J — QC Cloud Verdict")
print("=" * 60)

if len(df_results) > 0:
    # Pivot table
    pivot = df_results.pivot_table(
        values=["win_rate", "p_value"],
        index="coin",
        columns="horizon",
    )
    print("\nWin Rate (HAR-RV-J vs HAR Classic):")
    print(pivot["win_rate"].to_string(float_format="{:.1%}".format))
    print("\nP-values:")
    print(pivot["p_value"].to_string(float_format="{:.4f}".format))
    
    # Overall verdict
    total_beats = len(df_results[df_results["p_value"] < 0.05])
    total_combos = len(df_results)
    overall_p = stats.binomtest(
        df_results["j_wins"].sum(),
        df_results["n_obs"].sum(),
        alternative='greater'
    ).pvalue
    
    print(f"\nOverall: {total_beats}/{total_combos} BEATS at p<0.05")
    print(f"Cluster p-value: {overall_p:.2e}")
    print(f"\nLocal Python reference: 64/84 BEATS, p=7.9e-7")
    print(f"QC Cloud: {total_beats}/{total_combos} BEATS")
    
    # RV reconstruction mismatch check
    print(f"\nRV Mismatch Note:")
    for ticker, data in daily_data.items():
        print(f"  {ticker}: {data['n_days']} days, RV_mean={data['rv'].mean():.6f}")
else:
    print("\nNo results to report. Data may be unavailable on QC Cloud.")
    print("Check Section 1.2 for data availability.")
    print("Local Docker constraint: only SPY/QQQ/IWM equity data.")
    print("QC Cloud should provide crypto OHLCV hourly.")

print("\n" + "=" * 60)
print("END M12 HAR-RV-J QuantBook Research")
print("=" * 60)